# Amino acids: backbone or side chain?

Most amino acids keep their backbone in concentrated sulfuric acid. Some change their **side
chain**. Here you learn to tell the two apart.

| Part | What you do | Time |
|---|---|---|
| 1 | Read a stable amino acid (worked) | 10 min |
| 2 | Predict, then load your case | 10 min |
| 3 | Find the backbone signal | 15 min |
| 4 | Write your verdict and compare | 20 min |

Data: M. D. Seager, S. Seager, Bains and Petkowski, Zenodo [10.5281/zenodo.8381013](https://doi.org/10.5281/zenodo.8381013), CC BY 4.0. Paper: *Astrobiology* 24, 386 (2024).

In [ ]:
# Setup. In Colab this first installs the training package (about 30 seconds).
import importlib.util, os
if importlib.util.find_spec("nmrtrain") is None:
    %pip install -q "nmrtrain @ git+https://github.com/Rakiura-Inc/morning-star-nmr@main"
import matplotlib.pyplot as plt
import nmrtrain as nt

NOTEBOOK = "notebooks/03_amino_acid_side_chains.ipynb"
MODE = nt.portal.connect(NOTEBOOK)

## Part 1: A stable one, worked

The next cell stacks alanine in 98% acid at 1 day, 1 week and 1 month. Its alpha-H (on the
backbone carbon, about 4.6 ppm) and its methyl (about 2.0 ppm) do not move.

In [ ]:
ala = nt.load_series("8381013", "ALA", acid="K", nucleus="1H")
fig_ala, ax = nt.plot.stack(ala, 0.5, 9.5, norm=(0.5, 9.5), title="alanine in 98% acid")
plt.show()
for s in ala:
    pos, _ = nt.peaks.linewidth_hz(s, 6.82, 0.1)
    h = max((p["height"] for p in nt.peaks.pick(s, 0.5, 9.5) if abs(p["ppm"] - 6.82) < 0.05), default=0)
    print(f"{s.time_label:8s} NH3+ signal near {pos:.2f} ppm, height {h:.2f}")

### Trap: fading NH signals

The signal near 6.8 ppm shrinks. Those are NH3+ protons, and they swap with deuterium from the
acid. A fading NH signal is not breakdown.

## Part 2: Predict, then load your case

Asparagine and glutamine each carry an amide (C(=O)NH2) at the end of the side chain. Both were
recorded in 81% acid.

**Your turn**

In the next cell:

* Predict what happens to each: `"survives"`, `"side chain modified"`, `"degrades"` or `"unsure"`.
* Set `CASE` to the one you will read: `"ASN-H"` or `"GLN-H"`.

The cell after it stacks your case and prints its peak list at each time point.

In [ ]:
MY_PREDICTION = {"ASN-H": ..., "GLN-H": ...}
nt.record.predict("amino-acid", MY_PREDICTION)

CASE = "ASN-H"   # or "GLN-H"

In [ ]:
sample, acid = CASE.split("-")
ser = nt.load_series("8381013", sample, acid=acid, nucleus="1H")
fig_case, ax = nt.plot.stack(ser, 3.0, 6.5, norm=(3.0, 6.5),
                             title=f"{nt.io.NAMES[sample]} in {nt.io.ACID_PCT[acid]} acid")
plt.show()
traps = nt.checks.run_traps(ser, norm=(3.0, 6.5), lo=3.0, hi=6.5)
my_peaks = {s.time: nt.peaks.pick(s, 0.5, 9.5) for s in ser}
for t, pk in my_peaks.items():
    print(f"{t:7s}", [p["ppm"] for p in pk])
nt.portal.done("amino-acid-stack", figures=[fig_case]);

## Part 3: Find the backbone

The backbone signal is the alpha-H: the H on the carbon that carries both the NH3+ and the COOH.
In this acid it is usually the highest-ppm C-H signal of a simple amino acid.

**Your turn**

Find the alpha-H in your 1 day peak list above and set `BACKBONE_PPM` to its value.

In [ ]:
BACKBONE_PPM = ...   # a value from the 1 day list above

In [ ]:
if nt.record.revealed("amino-acid"):
    print(nt.checks.check_backbone(BACKBONE_PPM))

Now follow every signal over the three time points. Look for two things: whether the backbone
signal stays, and whether anything grows next to it. The next cell lists the peaks kept, lost and
new between the first and last spectrum.

In [ ]:
first, last = ser[0], ser[-1]
for k, v in nt.peaks.compare(my_peaks[first.time], my_peaks[last.time]).items():
    print(f"{k:5s}", ", ".join(f"{p:.2f}" for p in v) or "none")

## Part 4: Verdict and comparison

**Your turn**

Write your verdict in the next cell, using ppm values from your peak lists:

* `call`: `"survives"`, `"side chain modified"`, `"degrades"` or `"unsure"`
* `lost_ppm`: signals gone by 1 month
* `new_ppm`: signals that appear by 1 month
* `kept_ppm`: signals present at both times
* `product_guess`: what the side chain may have become, in words
* `confidence`: `"low"`, `"medium"` or `"high"`

In [ ]:
MY_VERDICT = {"call": "", "lost_ppm": [], "new_ppm": [], "kept_ppm": [], "product_guess": "", "confidence": ""}
print("\n".join(nt.checks.verdict_problems({CASE: MY_VERDICT}, [CASE])) or "Verdict format OK.")

In [ ]:
feedback = nt.checks.Report()
if nt.record.revealed("amino-acid"):
    key = nt.answers.load("amino_acids")
    c = nt.answers.case(key, CASE)
    feedback = nt.checks.check_verdict(MY_VERDICT, c).show("Your verdict")

### Look closer (optional): test one idea

If the side-chain amide turned into an acid, asparagine would become aspartic acid and glutamine
would become glutamic acid. The next cell lays your 1 month spectrum over that amino acid, recorded
in the same acid. Do any new peaks of yours line up with it? Then check your call against Table 1
of the paper (M. D. Seager et al. 2024, *Astrobiology* 24, 386).

In [ ]:
product = {"ASN": "ASP", "GLN": "GLU"}[sample]
prod = nt.load_series("8381013", product, acid=acid, nucleus="1H", times=["1month"])[0]
fig_prod, ax = nt.plot.stack([prod, last], 3.0, 6.5, norm=(3.0, 6.5),
                             labels=[f"{nt.io.NAMES[product]}, 1 month", f"{nt.io.NAMES[sample]}, 1 month"])
plt.show()

## Hand-in

The next cell saves your stack and `verdict.json` (your backbone signal, verdict, peak lists and
feedback) in `handin/`. Nothing is sent for review until you run `nt.portal.submit()`.

In [ ]:
record = nt.record.handin(
    "amino-acid-verdict", figures={"stack_" + CASE: fig_case},
    verdict={"case": CASE, "backbone_ppm": None if BACKBONE_PPM is ... else BACKBONE_PPM, "verdict": MY_VERDICT,
             "my_peaks_ppm": {t: nt.peaks.ppms(v) for t, v in my_peaks.items()},
             "feedback": [str(f) for f in feedback]},
    numbers={"case": CASE, "call": MY_VERDICT["call"] or "blank", "product": MY_VERDICT["product_guess"] or "blank"},
    question="amino-acid", traps=nt.checks.traps_flagged(traps))
# nt.portal.submit()